In [ ]:
# ── Cell 0: Config ──ACCOUNT = "mobicycle.us"LABELS = ["high", "medium", "low"]MODEL = "mistralai/Mistral-7B-Instruct-v0.2"MODEL_TYPE = "mistral"EPOCHS = 3BATCH_SIZE = 4LEARNING_RATE = 2e-4BLOCK_SIZE = 512LORA_RANK = 8LORA_ALPHA = 16LORA_DROPOUT = 0.05GRADIENT_ACCUMULATION = 4import json_cfg = {k: v for k, v in locals().items() if k.isupper()}with open('/content/config.json', 'w') as f:    json.dump(_cfg, f)print(f'Account: {ACCOUNT}')print(f'Labels: {LABELS}')print(f'Model: {MODEL}')print('Config saved.')

In [ ]:
# ── Cell 1: Install ──
!pip install -q torch transformers accelerate peft bitsandbytes datasets trl

In [ ]:
# ── Cell 2: GPU Check ──
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu} ({mem:.1f} GB)")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# PRIORITY CLASSIFICATION — 3 CLASSES
# ════════════════════════════════════════════════════════════════════
# The LoRA evaluates emails with common sense and assigns priority.
# These are guiding tendencies, NOT rules. The AI develops its own
# understanding of what high/medium/low means for each account.
#
# HIGH:
#   Government, regulatory, court, or enforcement senders
#   Active legal proceedings or enforcement actions
#   Financial obligations (overdue invoices, tax deadlines)
#   Security incidents, data breaches
#   Evidential material for legal matters
#   Client escalations or formal complaints
#
# MEDIUM:
#   Active client work and project communications
#   Business partner and supplier correspondence
#   Internal operational requests
#   Professional service provider communications
#   Scheduled deliverables and meeting coordination
#
# LOW:
#   Professional networking and relationship building
#   Industry newsletters and reports
#   Conference invitations and event notices
#   Optional professional development
#   Background reference material
#
# NOTE: Urgency (timeframe) is NOT part of priority.
# Urgency is assessed separately in Stage 2.
# ════════════════════════════════════════════════════════════════════
# ── Cell 3: Training Data (Black-and-White Examples) ──
# 120 examples across 3 priority classes. The model generalizes to gray areas.
import csv, os, random, json

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

# Format: (label, from_name, from_email, subject)
EXAMPLES = [
    # ──── HIGH (40) ────
    ("high", "Crown Court Listing", "listing@courts.gsi.gov.uk", "Hearing listed: Case MC-2025-0047"),
    ("high", "Environment Agency", "enforcement@environment-agency.gov.uk", "Compliance notice ref EA/2026/4412"),
    ("high", "ICO", "urgent@ico.org.uk", "Data breach notification: response required"),
    ("high", "Opposing Counsel", "litigation@lawfirm.co.uk", "Court order: witness statement required"),
    ("high", "HMRC", "penalties@hmrc.gov.uk", "Tax penalty appeal: action required"),
    ("high", "HSE", "investigations@hse.gov.uk", "Prohibition notice: cease operations pending review"),
    ("high", "Employment Tribunal", "listing@employmenttribunals.gov.uk", "Tribunal hearing listed"),
    ("high", "First-tier Tribunal", "listing@tribunals.gov.uk", "Directions compliance required"),
    ("high", "FCA", "enforcement@fca.org.uk", "Information requirement: response required"),
    ("high", "SFO", "enquiries@sfo.gov.uk", "Document preservation notice"),
    ("high", "Client Finance", "accounts@retailco.com", "Invoice MC-INV-4421 overdue 30 days"),
    ("high", "Companies House", "filing@companieshouse.gov.uk", "Annual accounts filing deadline approaching"),
    ("high", "Hiscox Insurance", "renewals@hiscox.com", "Professional indemnity renewal: cover lapsing"),
    ("high", "VAT Office", "vat@hmrc.gov.uk", "VAT return Q4 2025: filing deadline"),
    ("high", "Pension Regulator", "compliance@tpr.gov.uk", "Auto-enrolment re-declaration required"),
    ("high", "Zurich Insurance", "liability@zurich.co.uk", "Directors liability claim: response required"),
    ("high", "Bank", "compliance@businessbank.co.uk", "KYC refresh: updated documents required"),
    ("high", "Creditor", "collections@supplier.com", "Account on hold: outstanding balance"),
    ("high", "Admin Court", "orders@administrativecourt.gov.uk", "Judicial review: acknowledgment required"),
    ("high", "Local Authority", "enforcement@council.gov.uk", "Enforcement notice: remediation required"),
    ("high", "Bailiff Office", "enforcement@county-court.gov.uk", "Warrant of control: payment required"),
    ("high", "Upper Tribunal", "listing@uppertribunal.gov.uk", "Permission hearing: skeleton argument required"),
    ("high", "DEFRA", "returns@defra.gov.uk", "Packaging waste data return: submission required"),
    ("high", "Coroner", "court@coroner.gov.uk", "Inquest: your attendance is required"),
    ("high", "Client Legal", "legal@manufacturing.co.uk", "Settlement offer: response required"),
    ("high", "AML Team", "compliance@bankpartner.com", "Enhanced due diligence: documents needed"),
    ("high", "Council Tax", "business-rates@council.gov.uk", "Business rates appeal: evidence required"),
    ("high", "Landlord", "property@commercialreits.com", "Rent review notice: response required"),
    ("high", "Client HR", "hr@clientcorp.com", "Formal grievance: response required"),
    ("high", "Planning Inspector", "hearings@pins.gov.uk", "Inquiry: your evidence session scheduled"),
    ("high", "UKAS", "accreditation@ukas.com", "Surveillance audit: documentation required"),
    ("high", "Health Board", "contracts@nhstrust.nhs.uk", "Contract performance review: report required"),
    ("high", "Government Dept", "procurement@defra.gov.uk", "ITT response: submission required"),
    ("high", "Insurance Broker", "claims@broker.co.uk", "PI claim: insurer response required"),
    ("high", "CPS", "witnesses@cps.gov.uk", "Expert witness attendance required"),
    ("high", "Solicitor", "urgent@clientlaw.co.uk", "Injunction application: supporting evidence needed"),
    ("high", "Client CEO", "ceo@clientcorp.com", "Board meeting: need our report"),
    ("high", "Payroll Provider", "deadlines@payrollco.com", "Payroll submission: data required"),
    ("high", "HSE Inspections", "inspections@hse.gov.uk", "Workplace inspection findings: response required"),
    ("high", "Kohus EE", "info@kohus.ee", "Court proceedings: document submission required"),
    # ──── MEDIUM (40) ────
    ("medium", "Sarah Chen", "sarah@greencorp.com", "ESG assessment quotation request"),
    ("medium", "Project Lead", "pm@clientfirm.com", "Sprint review: need progress update"),
    ("medium", "Film Director", "director@prodco.com", "Script feedback needed before pre-production"),
    ("medium", "Partner Firm", "collab@enviro-consultants.com", "Joint proposal for council tender"),
    ("medium", "Internal PM", "projects@mobicycle.consulting", "Team standup agenda: add your updates"),
    ("medium", "New Lead", "enquiry@techstartup.com", "Interested in carbon footprint assessment"),
    ("medium", "Client Manager", "ops@foodmanufacturer.com", "Site visit to arrange for waste audit"),
    ("medium", "Subcontractor", "work@freelance-auditor.com", "Availability for site assessment?"),
    ("medium", "Council Officer", "planning@localcouncil.gov.uk", "Pre-application meeting: confirm attendance"),
    ("medium", "Marketing Team", "marketing@mobicycle.marketing", "Case study draft: review and approve"),
    ("medium", "Client Sustainability", "esg@bankinggroup.com", "Board ESG presentation: send deck draft"),
    ("medium", "IT Support", "helpdesk@mobicycle.tech", "Office 365 migration: team availability"),
    ("medium", "Recruitment", "jobs@talentfirm.com", "Interview scheduling: senior consultant candidates"),
    ("medium", "Accountant", "accounts@accountingfirm.co.uk", "Management accounts: review needed"),
    ("medium", "Client Request", "enquiries@energyco.com", "RFP for corporate sustainability strategy"),
    ("medium", "Internal HR", "hr@mobicycle.consulting", "Performance reviews: complete self-assessment"),
    ("medium", "Client Compliance", "compliance@pharmagroup.com", "Updated emissions data: please provide"),
    ("medium", "Supplier", "orders@labequipment.com", "Monitoring equipment delivery: confirm address"),
    ("medium", "Client Ops", "operations@logisticsgroup.com", "Supply chain mapping workshop: pick a date"),
    ("medium", "Conference Organiser", "speakers@esgconf.com", "Speaker slot confirmed: submit bio and abstract"),
    ("medium", "Standards Body", "iso@bsigroup.com", "ISO 14001 recertification: audit scheduled"),
    ("medium", "Client Sustainability 2", "esg@retailchain.com", "Annual sustainability report: data collection"),
    ("medium", "Training Provider", "courses@enviro-training.com", "Environmental auditor certification: next cohort"),
    ("medium", "Ellen MacArthur Foundation", "circular@emf.org", "Circular economy certification: application open"),
    ("medium", "IT Vendor", "renewals@softwareco.com", "Annual license renewal: review terms"),
    ("medium", "Property Agent", "viewings@commercialproperty.co.uk", "Office space options for expansion"),
    ("medium", "Service Provider", "contracts@cleaningco.com", "Office cleaning contract: review and renew"),
    ("medium", "GRI", "updates@globalreporting.org", "Updated GRI standards training: next session"),
    ("medium", "Fleet Manager", "vehicles@fleetco.com", "Company vehicle lease expiry: replacement options"),
    ("medium", "Professional Body", "cpd@rics.org", "CPD requirements: hours remaining"),
    ("medium", "Office Manager", "facilities@mobicycle.consulting", "Office refurbishment planning: input needed"),
    ("medium", "Charity Partner", "partnership@enviro-charity.org", "Corporate sponsorship renewal discussion"),
    ("medium", "Cloud Provider", "enterprise@cloudco.com", "Infrastructure contract review"),
    ("medium", "Research Institute", "collab@uni-research.ac.uk", "Joint research proposal: expression of interest"),
    ("medium", "Trade Association", "members@iema.net", "Annual member survey: please complete"),
    ("medium", "Website Host", "billing@hosting.com", "Domain and hosting renewal"),
    ("medium", "CIEEM", "events@cieem.net", "Spring conference registration"),
    ("medium", "Supplier Diversity", "programme@supplierdiversity.org", "Annual membership renewal"),
    ("medium", "Real Estate Agent", "commercial@estateagent.co.uk", "Lease review: renewal approaching"),
    ("medium", "McKinsey Report", "research@mckinsey.com", "New report: sustainability trends 2026"),
    # ──── LOW (40) ────
    ("low", "LinkedIn", "notifications@linkedin.com", "5 people viewed your profile this week"),
    ("low", "Alumni Network", "events@alumni-uni.ac.uk", "Summer reunion: register interest"),
    ("low", "Mentor", "personal@industry-veteran.com", "Coffee catch-up sometime soon?"),
    ("low", "Book Publisher", "review@publishinghouse.com", "New ESG textbook: potential endorsement"),
    ("low", "Volunteer Org", "help@green-volunteers.org", "Beach cleanup event: interested?"),
    ("low", "Professional Network", "digest@consultingnetwork.com", "Monthly digest: who is hiring"),
    ("low", "University", "careers@university.ac.uk", "Guest lecture invitation for autumn"),
    ("low", "Internal Social", "fun@mobicycle.consulting", "Summer team social: vote for activity"),
    ("low", "Charity Run", "fundraising@marathon-charity.org", "Corporate team entry for half marathon"),
    ("low", "Co-working Space", "membership@coworkspace.com", "New hot-desk packages available"),
    ("low", "Podcast Producer", "booking@businesspodcast.com", "Guest on our sustainability podcast?"),
    ("low", "Former Colleague", "hello@ex-colleague.com", "Catching up: moved to a new role"),
    ("low", "Thought Leadership", "editors@industrymag.com", "Invitation to write an opinion piece"),
    ("low", "Award Nomination", "entries@businessawards.com", "Nominate for Sustainability Consultancy of the Year"),
    ("low", "Wellness Provider", "corporate@wellnessprovider.com", "Corporate wellness programme: free trial"),
    ("low", "Photography", "headshots@photographer.com", "Team headshot session: booking slots"),
    ("low", "Stationery Supplier", "catalogue@officestuff.com", "New eco-friendly stationery range"),
    ("low", "Lunch Club", "reservations@lunchclub.co", "Networking lunch: environmental professionals"),
    ("low", "Gym Corporate", "corporate@fitness-club.com", "Corporate gym membership discount"),
    ("low", "Office Plants", "catalogue@officeplants.co.uk", "Biophilic office design service"),
    ("low", "Annual Review", "admin@mobicycle.consulting", "Save the date: strategy offsite December"),
    ("low", "Long-range Planning", "strategy@mobicycle.us", "2027 strategic plan: initial thoughts"),
    ("low", "PhD Student", "research@student.ac.uk", "Research request: consulting industry data"),
    ("low", "Conference 2027", "cfp@worldesgforum.com", "Call for papers: World ESG Forum"),
    ("low", "Retirement Plan", "pensions@pensionprovider.com", "Annual pension statement"),
    ("low", "Office Lease", "long-term@property.co.uk", "5-year lease option: new development"),
    ("low", "Alumni Magazine", "editors@alumni-mag.ac.uk", "Feature interview request"),
    ("low", "Government Strategy", "consultation@govuk.gov.uk", "25-year environment plan consultation"),
    ("low", "Archive Request", "records@nationalarchives.gov.uk", "Historical data request"),
    ("low", "Industry Book", "chapters@academic-press.com", "Invitation to contribute a chapter"),
    ("low", "Patent Application", "filings@ipo.gov.uk", "Patent status update"),
    ("low", "Sabbatical Policy", "hr@mobicycle.consulting", "Updated sabbatical policy"),
    ("low", "Trade Mission", "trade@ukgovernment.gov.uk", "Trade mission: expressions of interest"),
    ("low", "Tech Roadmap", "innovation@mobicycle.tech", "Technology platform roadmap: early input"),
    ("low", "CSR Report", "reporting@mobicycle.us", "Annual CSR report template"),
    ("low", "Climate Pledge", "updates@climatepledge.com", "Annual signatory progress review"),
    ("low", "Insurance Review", "annual@broker.co.uk", "Annual insurance portfolio review"),
    ("low", "IT Strategy", "planning@mobicycle.tech", "3-year IT infrastructure plan"),
    ("low", "Mentoring Programme", "programme@industry-body.org", "Mentoring cohort: applications open"),
    ("low", "Art Commission", "gallery@officearts.com", "Office artwork: new pieces available"),
]

# Generate train-priority.csv
random.shuffle(EXAMPLES)
os.makedirs('/content/data/priority', exist_ok=True)
train_path = '/content/data/priority/train-priority.csv'
with open(train_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['text'])
    for label, name, email, subject in EXAMPLES:
        prompt = (f'<s>[INST] Classify this email\'s priority as high, medium, or low.\n\n'
                  f'To: {ACCOUNT.replace(".", " ").title()} <enquiries@{ACCOUNT}>\n'
                  f'From: {name} <{email}>\n'
                  f'Subject: {subject} [/INST] {label}</s>')
        w.writerow([prompt])

# Verify distribution
from collections import Counter
counts = Counter(label for label, *_ in EXAMPLES)
print(f'Total: {len(EXAMPLES)} examples')
for label, count in sorted(counts.items()):
    print(f'  {label}: {count} ({count/len(EXAMPLES)*100:.0f}%)')
print(f'Saved to {train_path}')

In [ ]:
# ── Cell 4: Train ──
import sys, types, json, os, math

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))
    print('Config reloaded')

# Fix Colab triton bug
try:
    import triton
    if not hasattr(triton, 'ops'):
        triton.ops = types.ModuleType('triton.ops')
        sys.modules['triton.ops'] = triton.ops
except ImportError:
    pass

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import torch

project = f'priority-{ACCOUNT}'
data_dir = '/content/data/priority'
output_dir = f'/content/{project}'

print(f'Training: {project}')

# Load dataset
ds = load_dataset('csv', data_files=f'{data_dir}/train-priority.csv', split='train')
print(f'Loaded {len(ds)} examples')

steps_per_epoch = math.ceil(len(ds) / (BATCH_SIZE * GRADIENT_ACCUMULATION))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * 0.1)

# 4-bit quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = BLOCK_SIZE
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map={'': 0},
    torch_dtype=torch.float16, trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
print(f'Model loaded: {MODEL} (4-bit)')

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    gradient_checkpointing=True,
    max_grad_norm=0.0,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy='epoch',
    report_to='none',
)

trainer = SFTTrainer(
    model=model, train_dataset=ds, args=training_args,
    peft_config=lora_config, processing_class=tokenizer,
)

print('Starting training...')
trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'Done! Adapter saved to {output_dir}')

del model, trainer, tokenizer
torch.cuda.empty_cache()

In [ ]:
# ── Cell 5: Patch for Workers AI ──
import json, os

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

project = f'priority-{ACCOUNT}'
output_dir = f'/content/{project}'
config_path = os.path.join(output_dir, 'adapter_config.json')

with open(config_path) as f:
    config = json.load(f)
config['model_type'] = MODEL_TYPE
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

for name in ['adapter_model.safetensors', 'adapter_config.json']:
    path = os.path.join(output_dir, name)
    if os.path.exists(path):
        print(f'{name}: {os.path.getsize(path) / 1e6:.1f} MB')
    else:
        print(f'{name}: MISSING!')
print(f'r={config.get("r")}, alpha={config.get("lora_alpha")}, model_type={config.get("model_type")}')

In [ ]:
# ── Cell 6: Copy for Download ──
import shutil, os, json

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

project = f'priority-{ACCOUNT}'
output_dir = f'/content/{project}'
download_dir = f'/content/download/{project}'
os.makedirs(download_dir, exist_ok=True)

for name in ['adapter_model.safetensors', 'adapter_config.json']:
    src = os.path.join(output_dir, name)
    dst = os.path.join(download_dir, name)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'{dst} ({os.path.getsize(dst) / 1e6:.1f} MB)')
    else:
        print(f'MISSING: {src}')

print(f'\nDownload from: {download_dir}')
print(f'Deploy to: models/fine-tuning/PEFT/LoRA/priority/{ACCOUNT}/adapter/')